## Analisis inicial y Union
Primera parte del exploratorio, lo que quiero ver es como unir los 3 excels juntos y cuan mal están:

In [17]:
# Imports iniciales
from datetime import datetime, time, timedelta
from pathlib import Path
import pandas as pd

Los leo:

In [18]:
# Lo pongo así para que no se rompa el path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
# Acepta "Totalizador_Formateado" y "Totalizadores_Formateados"
archivos = sorted(RAW_DIR.glob("Totalizador*_Formateado*.csv"))
assert archivos, f"No hay CSV en {RAW_DIR}: correr primero src/procesado_raw.py"

SEP = " | "                   # separador "Hoja | columna" que pone el procesado
CLAVES = ["DIA", "HORA", "N_REPETICION"]
HOJAS_DESCARTAR = ["Auxiliar", "Seguimiento Dia"]
ERRORES_EXCEL = ["#VALUE!", "#DIV/0!", "#N/A", "#REF!", "#NAME?", "#NUM!", "#NULL!"]
LIMITE = 23 * 3600            # 23:00:00 en segundos desde medianoche


def hoja_de(col):
    """'Consolidado EE | Frio (Kw)' -> 'Consolidado EE'. None para DIA, HORA, N_REPETICION."""
    return col.split(SEP, 1)[0] if SEP in col else None


def leer_formateado(ruta):
    # low_memory=False: con ~400 columnas de tipos mezclados pandas tipa por bloques
    df = pd.read_csv(ruta, low_memory=False, na_values=ERRORES_EXCEL,
                     dtype={"HORA": "string"})
    # El procesado escribe DIA como dd/mm/aaaa: sin format, pandas invierte día y mes
    df["DIA"] = pd.to_datetime(df["DIA"], format="%d/%m/%Y")
    return df


dfs = {}
for archivo in archivos:
    print("Leyendo", archivo.name)
    dfs[archivo.name] = leer_formateado(archivo)

for nombre, df in dfs.items():
    print(nombre, df.shape)

Leyendo Totalizador_Formateado 2020_2022.csv
Leyendo Totalizador_Formateado 2021_2023.csv
Leyendo Totalizador_Formateado 2022_2023.csv
Totalizador_Formateado 2020_2022.csv (15621, 344)
Totalizador_Formateado 2021_2023.csv (15342, 424)
Totalizador_Formateado 2022_2023.csv (12017, 431)


In [19]:
def a_segundos(horas):
    """Serie de horas en texto ('23:59:00') -> segundos desde las 00:00. NaN si no se entiende."""
    return pd.to_timedelta(horas, errors="coerce").dt.total_seconds()


def formato(seg):
    return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"


def revisar_horas(df):
    """Horas posteriores a las 23:00:00 y valores de hora con formato raro."""
    horas = df["HORA"]
    segundos = a_segundos(horas)

    tarde = segundos[segundos > LIMITE]
    print("Horas > 23:00:00 (valor: cantidad de filas)")
    for seg, n in tarde.value_counts().sort_index().items():
        print(f"  {formato(int(seg))}: {n}")

    raros = horas[segundos.isna() & horas.notna()]
    if len(raros):
        print("Valores con formato no reconocido:")
        for valor, n in raros.astype(str).value_counts().items():
            print(f"  {valor!r}: {n}")


def revisar_dias(df, min_filas=24, max_filas=25):
    """Filas por día, y un df aparte con los días con menos de min_filas o más de max_filas."""
    conteo = df.groupby("DIA").size()

    print("Cantidad de días según filas por día:")
    print(conteo.value_counts().sort_index())

    dias_raros = conteo[(conteo < min_filas) | (conteo > max_filas)].index
    df_raros = df[df["DIA"].isin(dias_raros)].copy()
    return conteo, df_raros

Me ponga a ver como es el formato de las :59 

**queda justificar por que las :59**

In [20]:
for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    revisar_horas(df)


=== Totalizador_Formateado 2020_2022.csv ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 10
  23:59:00: 621
  23:59:01: 13
  23:59:59: 2
Valores con formato no reconocido:
  '01/01/1900': 24

=== Totalizador_Formateado 2021_2023.csv ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 9
  23:59:00: 606
  23:59:01: 5
  23:59:59: 1
Valores con formato no reconocido:
  '01/01/1900': 24

=== Totalizador_Formateado 2022_2023.csv ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 5
  23:59:00: 476
  23:59:01: 3
  23:59:59: 1


La mayoría de horas están bien, hay que tener en cuenta las 23:00:01 seguramente son duplicados de 23:00 y no de :59. 

In [21]:
conteos = {}
lista_raros = []

for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    conteos[nombre], df_raros = revisar_dias(df)
    lista_raros.append(df_raros)

raros = pd.concat(lista_raros, ignore_index=True)
print(raros.shape)


=== Totalizador_Formateado 2020_2022.csv ===
Cantidad de días según filas por día:
8        1
23       2
24       5
25     574
26      21
49       1
50       5
52       1
100      2
Name: count, dtype: int64

=== Totalizador_Formateado 2021_2023.csv ===
Cantidad de días según filas por día:
9        1
16       1
23       3
24       3
25     584
26       1
49       1
50       5
51       1
100      2
Name: count, dtype: int64

=== Totalizador_Formateado 2022_2023.csv ===
Cantidad de días según filas por día:
8        1
16       1
17       1
20       1
23       1
25     459
26       4
50       3
51       1
53       1
100      1
Name: count, dtype: int64
(2363, 450)


Hay como 100 días pegados. Saco ahora todos los valores que no sean 
- 23:59:00
- 23:59:01
- 23:59:59

In [22]:
def filtrar_2359(df):
    """Deja solo las filas con hora 23:59:xx (23:59:00, 23:59:01, 23:59:59)."""
    seg = a_segundos(df["HORA"].astype("string"))
    inicio = 23 * 3600 + 59 * 60          # 23:59:00
    fin = inicio + 59                     # 23:59:59
    return df[(seg >= inicio) & (seg <= fin)].copy()

In [23]:
dfs_2359 = {nombre: filtrar_2359(df) for nombre, df in dfs.items()}

# Veo para una tabla (solo EE, si no son ~400 columnas):
f = dfs_2359[nombre]
f.head(5)

,DIA,HORA,N_REPETICION,Consolidado KPI | EE Planta / Hl,Consolidado KPI | EE Elaboracion / Hl,Consolidado KPI | EE Bodega / Hl,Consolidado KPI | EE Cocina / Hl,Consolidado KPI | EE Envasado / Hl,Consolidado KPI | EE Linea 2 / Hl,Consolidado KPI | EE Linea 3 / Hl,...,Auxiliar | .21,Auxiliar | .22,Auxiliar | .23,Auxiliar | .24,Auxiliar | .25,Auxiliar | .26,Auxiliar | .27,Auxiliar | .28,Auxiliar | .29,Auxiliar | .30
24,2022-07-01,23:59:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,2022-07-02,23:59:00,0,8.040529,0.937245,0.768611,0.333626,1.720886,4.826019,2.298424,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,2022-07-03,23:59:00,0,6.574294,0.750594,0.577662,0.337017,1.539617,4.332178,1.922850,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,2022-07-04,23:59:00,0,7.766012,0.846567,0.736241,0.356453,1.890370,4.440273,1.540381,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,2022-07-05,23:59:00,0,6.916359,0.846227,0.683930,0.326098,1.640218,4.381815,1.549196,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Hojas presentes en cada archivo (según el prefijo "Hoja | ")
hojas_por_archivo = {n: {hoja_de(c) for c in df.columns} - {None} for n, df in dfs.items()}
todas = set.union(*hojas_por_archivo.values())
for n, h in hojas_por_archivo.items():
    print(f"{n}: {len(h)} hojas | faltan: {sorted(todas - h)}")

Totalizador_Formateado 2020_2022.csv: 15 hojas | faltan: ['Seguimiento Dia']
Totalizador_Formateado 2021_2023.csv: 16 hojas | faltan: []
Totalizador_Formateado 2022_2023.csv: 15 hojas | faltan: ['Seguimiento Dia']


Borro seguimiento día por que está en solo un excel y auxiliar **rellenar por que auxiliar, fabri sabía**

In [25]:
# Saco las hojas que no sirven (si no están, no pasa nada)
for n, df in dfs.items():
    cols = [c for c in df.columns if hoja_de(c) in HOJAS_DESCARTAR]
    dfs[n] = df.drop(columns=cols)
    print(f"{n}: saco {len(cols)} columnas de {HOJAS_DESCARTAR}")

Totalizador_Formateado 2020_2022.csv: saco 32 columnas de ['Auxiliar', 'Seguimiento Dia']
Totalizador_Formateado 2021_2023.csv: saco 35 columnas de ['Auxiliar', 'Seguimiento Dia']
Totalizador_Formateado 2022_2023.csv: saco 33 columnas de ['Auxiliar', 'Seguimiento Dia']


### Union de los datos

In [ ]:
partes = []
for nombre, df in dfs.items():
    f = filtrar_2359(df)
    f.insert(0, "ARCHIVO", nombre)
    partes.append(f)
    print(f"{nombre:45s} {len(f):5d} filas 23:59 | {f['DIA'].nunique()} días")

# Une por nombre de columna; las que no están en algún archivo quedan NaN
df_total = pd.concat(partes, ignore_index=True)
df_total = df_total.sort_values(["DIA", "HORA", "ARCHIVO", "N_REPETICION"], kind="stable").reset_index(drop=True)

print("Unido:", df_total.shape, df_total["DIA"].min().date(), "->", df_total["DIA"].max().date())


Totalizador_Formateado 2020_2022.csv            636 filas 23:59 | 609 días
Totalizador_Formateado 2021_2023.csv            612 filas 23:59 | 599 días
Totalizador_Formateado 2022_2023.csv            480 filas 23:59 | 471 días
Unido: (1728, 413) 2020-07-01 -> 2023-10-25
Días únicos: 1185
Días con más de una fila: 516
Días presentes en más de un archivo: 494


In [ ]:
SALIDA = ROOT / "data" / "processed" / "datos_base.csv"
df_total.to_csv(SALIDA, index=False, date_format="%Y-%m-%d")
print("Guardado en", SALIDA)

Guardado en c:\dev\Cerveceria-LabDatosII\data\processed\datos_base.csv


## Analisis del DF completo

In [ ]:
df_total = pd.read_csv(SALIDA, low_memory=False, parse_dates=["DIA"], dtype={"HORA": "string"})
print(df_total.shape)
df_total.head()

(1728, 413)


,ARCHIVO,DIA,HORA,N_REPETICION,Consolidado KPI | EE Planta / Hl,Consolidado KPI | EE Elaboracion / Hl,Consolidado KPI | EE Bodega / Hl,Consolidado KPI | EE Cocina / Hl,Consolidado KPI | EE Envasado / Hl,Consolidado KPI | EE Linea 2 / Hl,...,Consolidado Produccion | Unnamed: 18,Totalizadores Produccion | Hl Session IPA,Totalizadores Produccion | Id,Totalizadores Energia | Id,Totalizadores Agua | Id,Totalizadores Gas y Vapor | Id,Totalizadores Aire | Id,Totalizadores CO2 | Id,Totalizadores Efluentes | Id,"Totalizadores Glicol | Tot L3, L4 y Planta de CO2"
0,Totalizador_Formateado 2020_2022.csv,2020-07-01,23:59:00,0,642.727209,47.145349,69.023256,0.000000,13.813953,14.578784,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Totalizador_Formateado 2020_2022.csv,2020-07-02,23:59:00,0,7.767254,0.769609,0.798838,0.319229,2.358593,4.158962,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Totalizador_Formateado 2020_2022.csv,2020-07-03,23:59:00,0,8.801205,0.862593,0.835762,0.260924,1.985462,39.076667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Totalizador_Formateado 2020_2022.csv,2020-07-04,23:59:00,0,5.175639,0.439225,0.371077,0.258048,1.442114,4.348182,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Totalizador_Formateado 2020_2022.csv,2020-07-05,23:59:00,0,7.924665,0.802365,0.717787,0.301592,1.664726,5.125920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Para ver la cantidad de días únicos: 

In [28]:
por_dia = df_total.groupby("DIA").agg(filas=("ARCHIVO", "size"), archivos=("ARCHIVO", "nunique"))
print("Días únicos:", len(por_dia))
print("Días con más de una fila:", (por_dia["filas"] > 1).sum())
print("Días presentes en más de un archivo:", (por_dia["archivos"] > 1).sum())

Días únicos: 1185
Días con más de una fila: 516
Días presentes en más de un archivo: 494


Para ver los tipos de datos de los diferentes valores:

In [29]:
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Columns: 413 entries, ARCHIVO to Totalizadores Glicol | Tot L3, L4 y Planta de CO2
dtypes: datetime64[us](1), float64(401), int64(1), object(7), str(2), string(1)
memory usage: 5.4+ MB


Hay
- Datetime - es el día.
- String (nombre largo) - es la hora.

Quedan encontrar 
- 7 Objects
- 2 str

# DE ACA PARA ABAJO ARRANCA EL EDA, ESTO HAY QUE TRABAJAR Y PENSAR

Muy importante, hay 5 objetcs y 14 strings, hay que buscarlas y corregirlas seguramente. Primero igual tengo la sospecha que consolidado KPI tiene las columnas del resto de los valores:

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Si df_total tiene columnas MultiIndex (hoja, columna), las aplano a solo el nombre
if isinstance(df_total.columns, pd.MultiIndex):
    df_total.columns = df_total.columns.get_level_values(-1)

# Solo numéricas y con variación (las constantes dan falsos positivos)
X = df_total.select_dtypes("number")
X = X.loc[:, (X.nunique() > 1).values]
X = X.loc[:, ~X.columns.astype(str).str.startswith("Unnamed")]

# Etiqueta única por columna: "nombre [posición]"
X.columns = [f"{c} [{i}]" for i, c in enumerate(X.columns)]
print(X.shape)

(1728, 363)


In [ ]:
base = pd.Series(X.columns).str.replace(r"\s*\[\d+\]$", "", regex=True).str.lower().str.strip()
rep = base[base.duplicated(keep=False)]
print(rep.groupby(rep).apply(lambda s: list(X.columns[s.index])).to_string())

Series([], )


In [ ]:
grupos = defaultdict(list)
for col in X.columns:
    h = pd.util.hash_pandas_object(X[col].round(4), index=False).sum()
    grupos[h].append(col)

identicas = [cols for cols in grupos.values() if len(cols) > 1]
for cols in identicas:
    print(cols)
print(f"\n{len(identicas)} grupos de columnas idénticas")

['Consolidado Produccion | Hl Cerveza L2 [60]', 'Totalizadores Produccion | Hl Cerveza L2 [85]']
['Consolidado Produccion | Hl Cerveza L3 [61]', 'Totalizadores Produccion | Hl Cerveza L3 [86]']
['Consolidado Produccion | Hl Cerveza L4 [62]', 'Totalizadores Produccion | Hl Cerveza L4 [87]']
['Consolidado Produccion | Hl Cerveza L5 [63]', 'Totalizadores Produccion | Hl Cerveza L5 [88]']
['Consolidado Produccion | Cocimientos Diarios [64]', 'Totalizadores Produccion | Cocimientos Diarios [89]']
['Consolidado EE | KW Gral Planta [111]', 'Totalizadores Energia | KW Gral Planta [112]']
['Consolidado Agua | Red L1 y L2 [170]', 'Totalizadores Agua | Red L1 y L2 [177]']
['Consolidado Agua | FC L1 y L2 [171]', 'Totalizadores Agua | FC L1 y L2 [178]']
['Consolidado Agua | FC Lavadora L2 [172]', 'Totalizadores Agua | FC Lavadora L2 [203]']
['Consolidado Agua | Red L3 [173]', 'Totalizadores Agua | Red L3 [180]']
['Consolidado Agua | FC Lavadora L3 [174]', 'Totalizadores Agua | FC Lavadora L3 [201]'

In [ ]:
corr = X.corr()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pares = corr.where(mask).stack().reset_index()
pares.columns = ["a", "b", "r"]

umbral = 0.999                         # bajalo a 0.98 para ver las "demasiado parecidas"
pares = pares[pares["r"].abs() > umbral].sort_values("r", ascending=False)
print(len(pares), "pares con |r| >", umbral)
print(pares.to_string(index=False))

584 pares con |r| > 0.999
                                                   a                                                         b         r
                       Totalizadores Aire | Id [359]                        Totalizadores Efluentes | Id [361]  1.000000
                Totalizadores Gas y Vapor | Id [358]                        Totalizadores Efluentes | Id [361]  1.000000
                Totalizadores Gas y Vapor | Id [358]                             Totalizadores Aire | Id [359]  1.000000
          Consolidado GasVapor | Vapor_L5 (KG) [342]           Totalizadores Gas y Vapor | Vapor_L5 (KG) [346]  1.000000
                Consolidado KPI | Meta Aire L4 [331]               Consolidado KPI | Meta Aire Servicios [333]  1.000000
                Consolidado KPI | Meta Aire L3 [330]               Consolidado KPI | Meta Aire Servicios [333]  1.000000
                Consolidado KPI | Meta Aire L3 [330]                      Consolidado KPI | Meta Aire L5 [332]  1.000000
      

La verdad que c,laude hizo esto. Me voy a dormir y mañana lo hago/interpreto/pienso/reniego.